# Hybrid Architecture Decision Report

A hybrid approach combines retrieved evidence with learned or prompted behavior. Here the model receives both policy context and style examples, then returns a structured comparison that supports an architecture recommendation.


## What A Strategy Decision Must Contain

A useful architecture decision is not just a winner label. It should say what evidence was tested, which risks remain, and what trigger would justify revisiting the decision. For RAG and fine-tuning choices, the most important risks are freshness, citation auditability, behavior consistency, cost, latency, and maintenance effort.

The hybrid strategy is often attractive, but it should not be the default just because it sounds complete. Start with the simplest approach that satisfies the evaluation criteria. Add style conditioning or fine-tuning only when the measured failures point to behavior adaptation rather than missing evidence.


## Decision Criteria Definitions

**Freshness** means the system can reflect updated policy without retraining. RAG usually wins here because documents can be updated and re-indexed. **Auditability** means a reviewer can trace an answer back to evidence. Citations and source IDs support this. **Behavior consistency** means the system responds with the same tone, format, and refusal style across similar requests. Fine-tuning or strong examples can help here.

**Latency** is the time needed to answer. Retrieval and larger prompts can add overhead. **Cost** includes provider calls, embeddings, vector storage, training, and maintenance. **Maintenance effort** includes keeping documents current, reviewing examples, monitoring failures, and deciding when a strategy should be revisited.


### Exercise

Which risk does citation auditability address: freshness, grounding, or tone?


### Solution

Citation auditability mainly addresses grounding because reviewers can trace an answer back to retrieved evidence.


## Visual Map: Hybrid Strategy Combines Evidence And Behavior

The decision report compares retrieval quality, behavior consistency, operational cost, and the trigger for a future fine-tuning experiment.

**Embedding space visualization.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/PaCMAP_embeddings.png). This visual helps explain why retrieval evaluation is not only about the final answer: the geometry of examples and passages influences what evidence is available to the generator.

<img src="assets/hf-cookbook-pacmap-embeddings.png" alt="Embedding visualization" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

What two signals does the hybrid strategy combine?


### Solution

It combines retrieved evidence for grounding with style examples for response behavior.


## Load Dependencies And Shared Data

The setup is split into small cells so each responsibility is visible: imports first, then `.env` loading and fixture loading.


In [ ]:
import json
import math
import os
import re
from collections import Counter
from pathlib import Path
from pprint import pprint
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

The notebooks share one small support-policy fixture. Keeping it shared makes the strategy comparisons fair: each notebook sees the same documents, requests, and style examples.


In [ ]:
# Load optional provider settings from .env without overwriting shell variables.
load_dotenv(dotenv_path=Path(".env"), override=False)

DATA_PATH = Path("data") / "strategy_cases.json"
with DATA_PATH.open(encoding="utf-8") as file:
    DATA = json.load(file)

# Keep each fixture role explicit so later cells do not mix evidence, examples, and evaluation.
DOCUMENTS = DATA["documents"]
REQUESTS = DATA["support_requests"]
STYLE_EXAMPLES = DATA["style_examples"]
SCENARIOS = DATA["strategy_scenarios"]

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, and {len(STYLE_EXAMPLES)} style examples."
)

## Configure The Provider

The notebooks use the OpenAI-compatible client style. Groq is tried first by default; Mistral is the fallback when the preferred provider has no key.


In [ ]:
class ProviderConfig(BaseModel):
    """Provider settings for one OpenAI-compatible chat endpoint."""

    provider: str
    model: str
    base_url: str
    api_key: str | None = None
    dry_run: bool = True

Provider selection is intentionally explicit. This makes fallback behavior easy to inspect before any model call happens.


In [ ]:
def provider_order(preferred_provider: str | None = None) -> list[str]:
    """Return the preferred provider followed by the other supported provider."""
    requested = (preferred_provider or os.getenv("LLM_PROVIDER") or "groq").lower()

    if requested == "groq":
        return ["groq", "mistral"]
    if requested == "mistral":
        return ["mistral", "groq"]

    # Unknown values keep dry mode readable instead of failing during setup.
    return [requested]

The loader returns dry mode when no key is available. Dry mode is part of the teaching design: notebooks stay runnable during review and in classrooms without credentials.


Each provider has a tiny environment-to-config adapter. Keeping this separate makes the fallback loader shorter and easier to read.


In [ ]:
def provider_config_from_env(provider_name: str) -> ProviderConfig | None:
    """Return live provider settings when that provider has an API key."""
    if provider_name == "groq" and os.getenv("GROQ_API_KEY"):
        return ProviderConfig(
            provider="groq",
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY"),
            dry_run=False,
        )

    if provider_name == "mistral" and os.getenv("MISTRAL_API_KEY"):
        return ProviderConfig(
            provider="mistral",
            model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
            base_url="https://api.mistral.ai/v1",
            api_key=os.getenv("MISTRAL_API_KEY"),
            dry_run=False,
        )

    return None

The loader checks providers in order and falls back to dry mode when no key is available.


In [ ]:
def load_provider_config(preferred_provider: str | None = None) -> ProviderConfig:
    """Load live provider settings, or return a deterministic dry-run config."""
    # Check providers in priority order so fallback behavior is deterministic.
    for provider_name in provider_order(preferred_provider):
        config = provider_config_from_env(provider_name)
        if config is not None:
            return config

    # No key was found, so keep the notebook runnable without network calls.
    fallback_model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
    return ProviderConfig(
        provider=provider_order(preferred_provider)[0],
        model=fallback_model,
        base_url="dry-run",
    )

In [ ]:
CONFIG = load_provider_config()
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

if CONFIG.dry_run:
    print("Set GROQ_API_KEY or MISTRAL_API_KEY in .env to run live model calls.")

## Call The Model Or Use Dry Mode

The next cells isolate client creation, request assembly, and fallback behavior. That keeps provider plumbing separate from the lesson logic.


In [ ]:
def get_client(config: ProviderConfig) -> OpenAI | None:
    """Create a client only when a live API key is configured."""
    if config.dry_run or config.api_key is None:
        return None

    return OpenAI(api_key=config.api_key, base_url=config.base_url)

Request parameters are built in one small helper so response formatting can be added only where a notebook needs structured JSON.


In [ ]:
def completion_kwargs(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> dict[str, Any]:
    """Build provider request arguments for chat completions."""
    kwargs: dict[str, Any] = {
        "model": CONFIG.model,
        "messages": messages,
        "temperature": temperature,
    }

    # JSON mode is optional because not every provider/model combination needs it.
    if response_format is not None:
        kwargs["response_format"] = response_format

    return kwargs

`call_model` is the single gateway used by lesson cells. If live credentials are missing or a live call fails, the notebook returns a deterministic teaching response.


Live completion execution is separated from dry-mode routing so the failure handling is easy to follow.


In [ ]:
def live_completion_content(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> str:
    """Run the live chat completion and return message content."""
    client = get_client(CONFIG)
    assert client is not None

    completion = client.chat.completions.create(
        **completion_kwargs(
            messages, temperature=temperature, response_format=response_format
        )
    )
    return completion.choices[0].message.content or ""

`call_model` now only decides between dry output, live output, and live-call fallback.


In [ ]:
def call_model(
    messages: list[dict[str, str]],
    *,
    dry_response: str,
    temperature: float = 0.2,
    response_format: dict[str, str] | None = None,
) -> str:
    """Call the configured provider, or return deterministic dry output."""
    # Dry mode is the expected path for participants without provider credentials.
    if CONFIG.dry_run:
        return dry_response

    try:
        return live_completion_content(
            messages, temperature=temperature, response_format=response_format
        )
    except Exception as error:
        print(f"Live provider call failed: {error}")
        print("Using deterministic dry response so the notebook remains runnable.")
        return dry_response

## Build A Transparent Local Retriever

The retriever is deliberately simple: tokenize text, build term-frequency counters, score cosine similarity, then sort candidate documents.


In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase text and keep word-like tokens for a tiny local retriever."""
    return re.findall(r"[a-z0-9]+", text.lower())


def term_counts(text: str) -> Counter[str]:
    """Count token frequency so repeated domain terms increase similarity."""
    return Counter(tokenize(text))

Document vectors are computed once. In a production RAG stack this role is usually handled by an embedding model and vector database; here it stays inspectable.


In [ ]:
DOCUMENT_VECTORS = {
    document["doc_id"]: term_counts(document["title"] + " " + document["text"])
    for document in DOCUMENTS
}

Cosine similarity gives a compact relevance score between the query counter and each document counter. The search function keeps source metadata attached for later citations.


Cosine similarity is the scoring function for this transparent retriever.


In [ ]:
def cosine(left: Counter[str], right: Counter[str]) -> float:
    """Compute cosine similarity between sparse term-frequency counters."""
    shared_terms = set(left) & set(right)
    numerator = sum(left[term] * right[term] for term in shared_terms)
    left_norm = math.sqrt(sum(value * value for value in left.values()))
    right_norm = math.sqrt(sum(value * value for value in right.values()))

    # Empty vectors cannot be compared, so they receive no retrieval score.
    if left_norm == 0 or right_norm == 0:
        return 0.0

    return numerator / (left_norm * right_norm)

Search keeps score calculation separate from context formatting so each step can be inspected.


In [ ]:
def search_documents(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Return top documents with retrieval scores and citation metadata."""
    query_vector = term_counts(query)
    scored_documents = []

    for document in DOCUMENTS:
        # Keep the document fields and add a rounded score for inspection.
        score = cosine(query_vector, DOCUMENT_VECTORS[document["doc_id"]])
        scored_documents.append({**document, "score": round(score, 4)})

    return sorted(scored_documents, key=lambda item: item["score"], reverse=True)[
        :top_k
    ]

The generator receives compact context lines with explicit source IDs for citation checks.


In [ ]:
def format_context(rows: list[dict[str, Any]]) -> str:
    """Format retrieved rows as citeable prompt context."""
    return "\n".join(f"[{row['doc_id']}] {row['text']}" for row in rows)

Use `retrieve` as the public lesson name for document search. The implementation stays one line so participants can see that retrieval is just ranked search at this stage.


In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Lesson-friendly alias for the local document search helper."""
    return search_documents(query, top_k=top_k)

## Build Few-Shot Style Messages

Few-shot messages make behavior examples visible in the prompt. They do not update model weights.


In [ ]:
def style_messages(examples: list[dict[str, str]]) -> list[dict[str, str]]:
    """Convert approved examples into alternating user and assistant messages."""
    messages: list[dict[str, str]] = []

    for example in examples:
        # The user turn shows the incoming request pattern.
        messages.append({"role": "user", "content": example["user"]})
        # The assistant turn shows the response style we want to imitate.
        messages.append({"role": "assistant", "content": example["assistant"]})

    return messages


def compact_style_examples(examples: list[dict[str, str]]) -> str:
    """Create a readable style summary for prompts and reports."""
    lines = []
    for example in examples:
        lines.append(
            f"- intent={example['intent']} tone={example['tone']}: {example['assistant']}"
        )
    return "\n".join(lines)

## Build The Hybrid Prompt

The hybrid prompt includes the top retrieved policy records plus examples of the desired response behavior. This is the pattern to compare against RAG-only and style-only prompts.


The hybrid result schema stores separate grounding and style scores so the decision report does not collapse everything into one vague quality label.


In [ ]:
class StrategyResult(BaseModel):
    """Structured comparison record for one strategy output."""

    strategy: str
    answer: str
    source_ids: list[str]
    grounding_score: float = Field(ge=0, le=1)
    style_score: float = Field(ge=0, le=1)
    recommendation_note: str

Build the hybrid prompt with both retrieved evidence and style guidance. The evidence stays closest to the question.


In [ ]:
def hybrid_messages(
    request: dict[str, Any], evidence: list[dict[str, Any]]
) -> list[dict[str, str]]:
    """Create a prompt that combines style examples with citeable context."""
    context = format_context(evidence)
    return [
        {
            "role": "system",
            "content": "Use the style examples and answer only from context. Return JSON matching: strategy, answer, source_ids, grounding_score, style_score, recommendation_note.",
        },
        {
            "role": "user",
            "content": f"Style examples:\n{compact_style_examples(STYLE_EXAMPLES)}",
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {request['question']}",
        },
    ]

The dry response mirrors the fields expected from the provider, including a grounding score based on whether retrieval found the expected document.


In [ ]:
def dry_hybrid_response(request: dict[str, Any], evidence: list[dict[str, Any]]) -> str:
    """Create deterministic structured output for credential-free execution."""
    retrieved_ids = [item["doc_id"] for item in evidence]
    return json.dumps(
        {
            "strategy": "hybrid",
            "answer": f"{evidence[0]['text']} [{evidence[0]['doc_id']}]",
            "source_ids": [evidence[0]["doc_id"]],
            "grounding_score": 1.0 if request["expected_doc"] in retrieved_ids else 0.5,
            "style_score": 0.8,
            "recommendation_note": "Hybrid is useful when citations and repeatable support style are both required.",
        }
    )

The final helper calls the provider and validates structured output, with an explicit fallback for parse failures.


In [ ]:
def hybrid_answer(request: dict[str, Any]) -> StrategyResult:
    """Call the provider with both retrieved evidence and style examples."""
    evidence = retrieve(request["question"], top_k=3)
    # The live call sees both style guidance and retrieved policy context.
    raw = call_model(
        hybrid_messages(request, evidence),
        dry_response=dry_hybrid_response(request, evidence),
        response_format={"type": "json_object"},
    )

    try:
        return StrategyResult.model_validate_json(raw)
    except ValidationError:
        return StrategyResult(
            strategy="hybrid",
            answer=raw,
            source_ids=[],
            grounding_score=0.0,
            style_score=0.0,
            recommendation_note="Could not parse structured provider output.",
        )

Run one hybrid call before building the strategy table so the output shape is visible.


In [ ]:
result = hybrid_answer(REQUESTS[2])
pprint(result.model_dump())

In [ ]:
# Exercise: inspect which source ID the hybrid answer used.
# hybrid_source_ids = ...


In [ ]:
# Solution
hybrid_source_ids = result.source_ids
print(hybrid_source_ids)

## Compare Strategies With Real Calls Available

The scoring table separates quality signals from operational assumptions. In dry mode, quality values are deterministic; with provider keys, the hybrid answer comes from the model.


## Interpreting The Comparison Table

The comparison table is intentionally small, but it models a real architecture review habit: keep different quality signals separate until the decision step. Retrieval grounding answers “did we bring the right evidence?” Style score answers “did the response match the desired behavior?” Operational notes answer “is this strategy maintainable enough to start with?”

If the table shows strong grounding and weak style, improve prompts or examples before scheduling a fine-tuning project. If the table shows weak grounding, fix retrieval before changing model behavior. Fine-tuning should be a response to measured behavior failures, not a shortcut around missing evidence.


## Example Strategy Recommendations

If retrieval grounding is high and style is acceptable, recommend RAG-only because it is simpler and easier to maintain. If retrieval grounding is high but tone is inconsistent, recommend a hybrid prompt with better examples before scheduling fine-tuning. If retrieval grounding is low, recommend improving chunking, query rewriting, or the corpus before changing model behavior.

Fine-tuning becomes a stronger recommendation only after repeated behavior failures are measured on held-out examples. Even then, it should be compared against a stronger prompt baseline. The decision report should explain what was tried, why the chosen path is sufficient for now, and what signal would trigger the next experiment.


In [ ]:
def retrieval_grounding_score(request: dict[str, Any], top_k: int = 3) -> float:
    """Score whether retrieval includes the expected source."""
    retrieved_ids = [
        item["doc_id"] for item in retrieve(request["question"], top_k=top_k)
    ]
    return 1.0 if request["expected_doc"] in retrieved_ids else 0.0


# Compare strategy signals per held-out request rather than relying on one example.
strategy_rows = []
for request in REQUESTS:
    hybrid = hybrid_answer(request)
    strategy_rows.append(
        {
            "request_id": request["request_id"],
            "rag_grounding": retrieval_grounding_score(request),
            "style_only_grounding": 0.0,
            "hybrid_grounding": hybrid.grounding_score,
            "hybrid_style": hybrid.style_score,
        }
    )
pprint(strategy_rows)

In [ ]:
# Exercise: count how many held-out requests were compared.
# compared_request_count = ...


In [ ]:
# Solution
compared_request_count = len(strategy_rows)
print(compared_request_count)

## Produce The Decision Report

The decision should state a starting architecture, the evidence for that choice, and what would justify moving to actual fine-tuning later.


Build decision-review messages separately from the dry response so the report structure is visible.


In [ ]:
decision_messages = [
    {
        "role": "system",
        "content": "You are an LLM architecture reviewer. Return concise JSON with recommended_strategy, rationale, and revisit_trigger.",
    },
    {
        "role": "user",
        "content": f"Use these strategy rows to decide whether to start with RAG-only, fine-tuning-only, or hybrid:\n{json.dumps(strategy_rows, indent=2)}",
    },
]

The dry decision summarizes the intended architectural conclusion.


In [ ]:
dry_decision = json.dumps(
    {
        "recommended_strategy": "RAG-first, then hybrid if style evaluation fails",
        "rationale": "Fresh policy evidence and citations are mandatory. Style adaptation is useful, but should be added only if RAG prompts do not meet tone and refusal expectations.",
        "revisit_trigger": "Collect enough labeled style failures to justify a real LoRA or hosted fine-tuning experiment.",
    },
    indent=2,
)

Call the provider or dry-mode fallback for the final decision report.


In [ ]:
raw_decision = call_model(
    decision_messages,
    dry_response=dry_decision,
    response_format={"type": "json_object"},
)
print(raw_decision)

### Exercise

What evidence would justify moving from prompt improvements to real fine-tuning?


### Solution

Repeated style or formatting failures on held-out evaluation examples would justify a real fine-tuning experiment.


In [ ]:
next_experiment = {
    "keep": "RAG evidence path",
    "try_before_fine_tuning": [
        "stronger system prompt",
        "more few-shot style examples",
        "structured response schema",
    ],
    "fine_tune_when": "style failures persist on held-out evaluation despite prompt improvements",
}
pprint(next_experiment)